In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

### Import and visualize data structure:

In [ ]:
path_otu = "../data/raw/otu_HMP2_16S.csv"
path_meta = "../data/raw/meta_HMP2.csv"
path_taxonomy = "../data/raw/taxonomy_HMP2_16S.csv"

otu = pd.read_csv(path_otu, index_col=0)
meta = pd.read_csv(path_meta, index_col=0)
taxonomy = pd.read_csv(path_taxonomy, index_col=0)

### Check that the data has expected dimensions:

In [ ]:
assert otu.index.equals(meta.index), "OTU rows and metadata rows do not match"
assert otu.columns.equals(
    taxonomy.index
), "OTU columns and taxa rows do not match"

### Dimensions of the datasets

In [ ]:
print(f"OTU shape: {otu.shape}")
print(f"Metadata shape: {meta.shape}")
print(f"Taxonomy shape: {taxonomy.shape}")
print()
print(f"Number of subjects: {meta['SubjectID'].nunique()}")

### Print the start of each dataset

In [ ]:
otu.head()

In [ ]:
meta.head()

In [ ]:
taxonomy.head()

### How many samples per subject

In [ ]:
meta["SubjectID"].value_counts().hist(bins=70)
plt.xlabel("Samples per subject")
plt.ylabel("Number of subjects")
plt.show()

Subject 69-001 has the most samples by a great margin

### How many total reading per sample

In [ ]:
sample_total_abundance = otu.sum(axis=1)

plt.hist(sample_total_abundance, bins=30)
plt.xlabel("Total reads per sample")
plt.ylabel("Number of samples")
plt.title("Sequencing depth")
plt.show()

### Most common taxa:

In [ ]:
taxa_total_abundance = otu.sum(
    axis=0
)  # Sum for each otu the total number of absolute abundance
taxa_total_abundance_sorted = taxa_total_abundance.sort_values(
    ascending=False
)  # Sort the taxa
top_taxa = taxa_total_abundance_sorted.head(20)

plt.figure(figsize=(10, 5))
top_taxa.plot(kind="bar")
plt.ylabel("Total abundance")
plt.title("Top 20 taxa")
plt.show()

### Name of most common OTUs

In [ ]:
# Crea una tabella con abbondanza e tassonomia
top_table = taxonomy.loc[top_taxa.index].copy()
top_table["Total abundance"] = top_taxa.values

# Metti l'abbondanza come prima colonna
top_table.insert(0, "Total abundance", top_table.pop("Total abundance"))

print(top_table.head())

### Relative abundance in different samples from every patient for the top 20 taxa

In [ ]:
from metagenomes_simulation.data_processing import (
    absolute_to_relative_abundance,
)

# Transform absolute to relative abundance
otu_rel = absolute_to_relative_abundance(otu)

# Measure the mean relative abundance for each taxon on every sample
average_abundance = otu_rel.mean(axis=0)
# Sort
average_abundance_sorted = average_abundance.sort_values(ascending=False)
# Select the top 20
top_relative_abundances = average_abundance_sorted.head(20)

# Plot
otu_rel[top_relative_abundances.index].plot(
    kind="area", stacked=True, figsize=(12, 5)
)
plt.ylabel("Relative abundance")
plt.show()

### Temporal evolution of relative abundance in a single patient

In [ ]:
# Choose the patient with the most number of samples
subject_id = "69-001"

# Isolate the subject samples
meta_subject = meta[meta["SubjectID"] == subject_id]

# Sort by time
meta_subject_sorted = meta_subject.sort_values("DateZero")

# Select subject OTUs
otu_subject = otu_rel.loc[meta_subject.index]

# Find the most abundant OTUs in the subject
average_abundance = otu_subject.mean(axis=0)
top_relative_abundances = average_abundance.sort_values(ascending=False).head(
    5
)

# Plot the first 5 most abundant OTUs
otu_subject[top_relative_abundances.index].plot(
    kind="area",
    stacked=True,
    figsize=(12, 5),
)

plt.xlabel("Samples from earliest to latest")
plt.ylabel("Relative abundance")
plt.title(f"Temporal evolution of subject {subject_id}")
plt.show()